# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pravu-19/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found. Add it to Colab Secrets as HF_TOKEN.")

con = duckdb.connect()

warehouse = "hf://datasets/FlyRank/internship-warehouse"

con.execute("DROP SECRET IF EXISTS hf_secret")
con.execute("""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN ?
    )
""", [HF_TOKEN])

PERF_GLOB = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=*/data_0.parquet"
)

print("Hugging Face authentication: OK")
print("Warehouse: FlyRank/internship-warehouse")

Hugging Face authentication: OK
Warehouse: FlyRank/internship-warehouse


## 1. Ranked actions + reason codes

### Ranked actions + reason codes

The playbook converts the validated CTR prediction into a practical, ranked content-review queue. The ranking is intended to identify where a human should investigate first, not to automatically change content.

The highest-priority actions should combine meaningful historical visibility with a useful model signal. Very low-volume observations are treated cautiously because the validation audit showed that the largest individual errors were concentrated in cases with only a few impressions. Therefore, a large prediction error alone is not sufficient evidence for an action.

I use these reason codes:

* **HIGH_OPPORTUNITY** — meaningful historical visibility combined with a model signal that suggests the page deserves review.
* **CTR_GAP** — predicted future CTR is materially different from the historical CTR, creating a review opportunity.
* **POSITION_OPPORTUNITY** — historical average position suggests the page is visible enough that improving relevance, title, or content quality may be worth investigating.
* **REFRESH_CANDIDATE** — the page has enough historical activity to justify checking whether content freshness or alignment should be reviewed.
* **LOW_VOLUME_CAUTION** — limited historical impressions make the recommendation less reliable; human review is required before prioritization.
* **STABLE / LOW_PRIORITY** — the model does not provide a strong enough signal to justify immediate action.

The queue is therefore a prioritization aid. A high rank means "review this earlier," not "make this change automatically."


In [5]:
# ============================================================
# ML-10 SECTION 1 — BUILD RANKED ACTION QUEUE
# ============================================================

import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# ------------------------------------------------------------
# 1. Setup
# ------------------------------------------------------------

con = duckdb.connect()

warehouse = "hf://datasets/FlyRank/internship-warehouse"

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add your Hugging Face token "
        "to Colab Secrets as HF_TOKEN."
    )

con.execute("DROP SECRET IF EXISTS hf_secret")
con.execute("""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN ?
    )
""", [HF_TOKEN])

PERF_GLOB = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=*/data_0.parquet"
)

print("Hugging Face authentication: OK")

Hugging Face authentication: OK


In [6]:
# ------------------------------------------------------------
# 2. Recreate the validated ML-08/ML-09 feature table
# ------------------------------------------------------------

model_df = con.execute(f"""
WITH base AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        CAST(gsc_impressions AS DOUBLE) AS impressions,
        CAST(gsc_clicks AS DOUBLE) AS clicks,
        CAST(gsc_avg_position AS DOUBLE) AS avg_position,
        CAST(ga4_pageviews AS DOUBLE) AS pageviews,
        CAST(ga4_sessions AS DOUBLE) AS sessions,
        CAST(ga4_users AS DOUBLE) AS users,
        CAST(ga4_engaged_sessions AS DOUBLE) AS engaged_sessions,
        CAST(scroll_events AS DOUBLE) AS scroll_events
    FROM read_parquet(
        '{PERF_GLOB}',
        hive_partitioning=true
    )
    WHERE report_date >= '2026-01-01'
      AND report_date < '2026-07-01'
      AND gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_impressions > 0
),

historical AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(impressions) AS hist_impressions,
        SUM(clicks) AS hist_clicks,
        SUM(clicks) / NULLIF(SUM(impressions), 0) AS hist_ctr,
        AVG(avg_position) AS hist_avg_position,
        SUM(pageviews) AS hist_pageviews,
        SUM(sessions) AS hist_sessions,
        SUM(users) AS hist_users,
        SUM(engaged_sessions) AS hist_engaged_sessions,
        SUM(scroll_events) AS hist_scroll_events
    FROM base
    WHERE report_date >= '2026-01-01'
      AND report_date < '2026-05-01'
    GROUP BY client_hash_id, content_hash_id
),

may_target AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(impressions) AS may_impressions,
        SUM(clicks) AS may_clicks,
        SUM(clicks) / NULLIF(SUM(impressions), 0) AS may_ctr
    FROM base
    WHERE report_date >= '2026-05-01'
      AND report_date < '2026-06-01'
    GROUP BY client_hash_id, content_hash_id
),

june_target AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(impressions) AS june_impressions,
        SUM(clicks) AS june_clicks,
        SUM(clicks) / NULLIF(SUM(impressions), 0) AS june_ctr
    FROM base
    WHERE report_date >= '2026-06-01'
      AND report_date < '2026-07-01'
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    h.*,
    m.may_impressions,
    m.may_clicks,
    m.may_ctr,
    j.june_impressions,
    j.june_clicks,
    j.june_ctr
FROM historical h
JOIN may_target m
    USING (client_hash_id, content_hash_id)
JOIN june_target j
    USING (client_hash_id, content_hash_id)
WHERE m.may_ctr IS NOT NULL
  AND j.june_ctr IS NOT NULL
""").fetchdf()

print("Modeling rows:", len(model_df))
print("Unique clients:", model_df["client_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 160595
Unique clients: 46


In [7]:
# ------------------------------------------------------------
# 3. Train the validated Random Forest
# ------------------------------------------------------------

features = [
    "hist_impressions",
    "hist_clicks",
    "hist_ctr",
    "hist_avg_position",
    "hist_pageviews",
    "hist_sessions",
    "hist_users",
    "hist_engaged_sessions",
    "hist_scroll_events"
]

X = (
    model_df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y_may = model_df["may_ctr"].astype(float)

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y_may)

model_df["predicted_ctr"] = rf.predict(X)

print("Random Forest trained.")

Random Forest trained.


In [8]:
# ------------------------------------------------------------
# 4. Build practical action signals
# ------------------------------------------------------------

queue = model_df[
    [
        "client_hash_id",
        "content_hash_id",
        "hist_impressions",
        "hist_clicks",
        "hist_ctr",
        "hist_avg_position",
        "predicted_ctr"
    ]
].copy()

queue["ctr_gap"] = (
    queue["predicted_ctr"] - queue["hist_ctr"]
)

queue["abs_ctr_gap"] = queue["ctr_gap"].abs()

# Volume confidence:
# very low-volume pages are explicitly down-weighted.
queue["volume_factor"] = np.minimum(
    np.log1p(queue["hist_impressions"]) / np.log1p(1000),
    1.0
)

queue["volume_factor"] = queue["volume_factor"].clip(0, 1)

# Opportunity score is directional and intended only for ranking.
queue["action_score"] = (
    queue["abs_ctr_gap"] * queue["volume_factor"]
)

# ------------------------------------------------------------
# 5. Reason codes
# ------------------------------------------------------------

def reason_code(row):
    if row["hist_impressions"] <= 10:
        return "LOW_VOLUME_CAUTION"

    if row["abs_ctr_gap"] >= queue["abs_ctr_gap"].quantile(0.90):
        return "CTR_GAP"

    if row["hist_avg_position"] <= 10:
        return "POSITION_OPPORTUNITY"

    if row["hist_impressions"] >= queue["hist_impressions"].quantile(0.75):
        return "HIGH_OPPORTUNITY"

    return "STABLE_LOW_PRIORITY"


queue["reason_code"] = queue.apply(reason_code, axis=1)

# ------------------------------------------------------------
# 6. Human action mapping
# ------------------------------------------------------------

action_map = {
    "HIGH_OPPORTUNITY": "Review content performance and prioritize human investigation",
    "CTR_GAP": "Review search intent, title/snippet alignment, and content relevance",
    "POSITION_OPPORTUNITY": "Review page relevance and SERP-facing content elements",
    "REFRESH_CANDIDATE": "Review content freshness and update need",
    "LOW_VOLUME_CAUTION": "Do not prioritize automatically; validate with additional evidence",
    "STABLE_LOW_PRIORITY": "Monitor; no immediate content action"
}

queue["recommended_action"] = (
    queue["reason_code"].map(action_map)
)

# Rank highest action score first.
queue = queue.sort_values(
    ["action_score", "hist_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

display(
    queue[
        [
            "rank",
            "hist_impressions",
            "hist_ctr",
            "predicted_ctr",
            "abs_ctr_gap",
            "hist_avg_position",
            "reason_code",
            "recommended_action"
        ]
    ].head(20)
)

,rank,hist_impressions,hist_ctr,predicted_ctr,abs_ctr_gap,hist_avg_position,reason_code,recommended_action
0,1,27.0,0.555556,0.031878,0.523677,2.366667,CTR_GAP,"Review search intent, title/snippet alignment,..."
1,2,2.0,1.000000,0.010183,0.989817,0.000000,LOW_VOLUME_CAUTION,Do not prioritize automatically; validate with...
2,3,2.0,1.000000,0.014724,0.985276,4.500000,LOW_VOLUME_CAUTION,Do not prioritize automatically; validate with...
3,4,2.0,1.000000,0.037038,0.962962,1.000000,LOW_VOLUME_CAUTION,Do not prioritize automatically; validate with...
4,5,44.0,0.318182,0.059173,0.259009,20.541667,CTR_GAP,"Review search intent, title/snippet alignment,..."
5,6,16.0,0.375000,0.051803,0.323197,6.114286,CTR_GAP,"Review search intent, title/snippet alignment,..."
6,7,3.0,0.666667,0.024823,0.641843,28.333333,LOW_VOLUME_CAUTION,Do not prioritize automatically; validate with...
7,8,11.0,0.363636,0.021230,0.342406,3.888889,CTR_GAP,"Review search intent, title/snippet alignment,..."
8,9,10.0,0.400000,0.048633,0.351367,7.555556,LOW_VOLUME_CAUTION,Do not prioritize automatically; validate with...
9,10,4.0,0.500000,0.012651,0.487349,5.833333,LOW_VOLUME_CAUTION,Do not prioritize automatically; validate with...


## 2. Intended use and limits

This playbook is intended for content and SEO analysts who need a repeatable way to decide which content should be reviewed first. It converts historical performance signals and the validated Random Forest output into a ranked decision-support queue.

The model should be used to prioritize investigation, not to predict guaranteed outcomes from a content change.

The June 2026 forward test is the main evidence used here. The Random Forest measured MAE of 0.004764 compared with 0.005440 for the historical-CTR baseline, which is a 12.43% lower measured MAE in this specific forward-test setup. This is an observed, directional result and should not be interpreted as proof of causal impact or guaranteed future improvement.

The model is limited by the available features and time period. It uses historical impressions, clicks, CTR, average position, pageviews, sessions, users, engaged sessions, and scroll events. It does not directly observe search intent, content quality, business value, SERP features, algorithm changes, or the reason a page was selected for a content update.

Low-volume observations require additional caution. The validation audit showed that extreme individual errors can occur when a page has very few impressions, because one click can create an unstable observed CTR.

The playbook is therefore a human decision-support layer, not a production automation system.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Human review rules

Every recommended action must be reviewed by a person before implementation.

The reviewer should check:

1. Whether the page is relevant to the intended search topic.
2. Whether the observed traffic volume is sufficient to make the signal meaningful.
3. Whether the page's historical position and CTR make the recommendation plausible.
4. Whether the content is outdated, incomplete, duplicated, or misaligned with user intent.
5. Whether there are business, editorial, legal, or brand constraints not represented in the dataset.
6. Whether another recent change could explain the observed performance.
7. Whether the proposed action has a reasonable expected value compared with its implementation cost.

### No-go list

The model should **not** automatically:

* publish or rewrite content;
* change titles or metadata without review;
* delete or redirect pages;
* change important business or legal claims;
* make decisions about sensitive or high-impact content;
* treat low-volume CTR spikes as reliable opportunities;
* claim that a recommended action will cause higher CTR;
* override editorial or subject-matter expertise;
* trigger production changes directly from the ranking.

The correct workflow is:

**model signal → ranked queue → human review → optional experiment/change → measure outcome.**

This keeps the playbook practical while avoiding unsupported automation.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

The playbook should be checked regularly rather than assumed to remain valid indefinitely.

Monitor:

* prediction MAE on a new future period;
* comparison with the historical-CTR baseline;
* distribution of historical CTR, impressions, clicks, and average position;
* percentage of recommendations coming from low-volume observations;
* changes in the number and type of content records;
* whether the highest-ranked recommendations continue to receive meaningful human approval.

### Suggested retrain triggers

A retraining review should be considered when:

1. New forward-period MAE becomes materially worse than the validated June result.
2. The Random Forest no longer performs better than the historical-CTR baseline in a comparable forward test.
3. Feature distributions change substantially from the training period.
4. Search or measurement conditions change enough that January–April historical relationships may no longer represent current behavior.
5. The content population changes materially.
6. Human reviewers repeatedly reject the model's highest-ranked recommendations.

These are monitoring and review triggers, not claims that a specific threshold has already been validated. The current evidence supports a time-aware forward test, but it does not establish a universal retraining threshold.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [9]:
# ============================================================
# ML-10 SECTION 5 — EXPORTS
# ============================================================

from pathlib import Path
import json

ROOT = Path.cwd()

OUTPUT_DIR = ROOT / "work" / "outputs"
FIGURE_DIR = ROOT / "work" / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Ranked queue
# ------------------------------------------------------------

export_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "hist_impressions",
    "hist_clicks",
    "hist_ctr",
    "hist_avg_position",
    "predicted_ctr",
    "ctr_gap",
    "abs_ctr_gap",
    "volume_factor",
    "action_score",
    "reason_code",
    "recommended_action"
]

action_queue = queue[export_columns].copy()

queue_path = OUTPUT_DIR / "action_queue.csv"
action_queue.to_csv(queue_path, index=False)

print(f"Exported ranked queue: {queue_path}")

# ------------------------------------------------------------
# Paper-ready metrics
# ------------------------------------------------------------

baseline_mae = 0.005440107943592033
rf_mae = 0.004763730329129472

relative_difference = (
    (baseline_mae - rf_mae) / baseline_mae
) * 100

paper_metrics = {
    "dataset": "FlyRank/internship-warehouse",
    "feature_window": "2026-01-01 to 2026-04-30",
    "training_target": "2026-05-01 to 2026-05-31",
    "forward_test": "2026-06-01 to 2026-06-30",
    "baseline_mae": baseline_mae,
    "random_forest_mae": rf_mae,
    "relative_mae_difference_percent": relative_difference,
    "interpretation": "Observed directional decision-support result; not causal."
}

metrics_path = OUTPUT_DIR / "action_playbook_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(paper_metrics, f, indent=2)

print(f"Exported metrics: {metrics_path}")

# ------------------------------------------------------------
# Reason-code summary
# ------------------------------------------------------------

reason_summary = (
    action_queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="count")
)

reason_path = OUTPUT_DIR / "action_reason_summary.csv"
reason_summary.to_csv(reason_path, index=False)

display(reason_summary)

print("All ML-10 exports completed.")

Exported ranked queue: /content/work/outputs/action_queue.csv
Exported metrics: /content/work/outputs/action_playbook_metrics.json


,reason_code,count
0,POSITION_OPPORTUNITY,67523
1,STABLE_LOW_PRIORITY,58174
2,HIGH_OPPORTUNITY,13602
3,CTR_GAP,13033
4,LOW_VOLUME_CAUTION,8263


All ML-10 exports completed.


In [10]:
# ============================================================
# ML-10 SELF-CHECK
# ============================================================

assert len(action_queue) > 0
assert action_queue["rank"].iloc[0] == 1
assert action_queue["rank"].is_monotonic_increasing

assert set(action_queue["reason_code"].dropna()).issubset({
    "HIGH_OPPORTUNITY",
    "CTR_GAP",
    "POSITION_OPPORTUNITY",
    "REFRESH_CANDIDATE",
    "LOW_VOLUME_CAUTION",
    "STABLE_LOW_PRIORITY"
})

assert queue_path.exists()
assert metrics_path.exists()
assert reason_path.exists()

print("ML-10 self-check: PASSED")
print("Ranked queue exported.")
print("Metrics exported.")
print("Reason-code summary exported.")
print("Human review / no-go rules documented.")
print("No production automation enabled.")

ML-10 self-check: PASSED
Ranked queue exported.
Metrics exported.
Reason-code summary exported.
Human review / no-go rules documented.
No production automation enabled.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.